# Cost-Savings Benchmark — `dynamic-model-router`

**Question:** what does it actually save you to route per-task instead of always calling `gpt-4o`?

**Method:** classify 1,000 prompts drawn from public benchmarks (MT-Bench, MMLU, HumanEval, ShareGPT) and compute:
1. **$ baseline** — total cost if every prompt → `gpt-4o`.
2. **$ routed** — total cost when each prompt → `Router().classify().model_name`.
3. **Tier distribution** — % of prompts hitting LOW / MEDIUM / HIGH.
4. **Quality parity** — sampled LLM-as-judge on a random 100 (optional, requires API key).

**Cost mode:** this notebook runs **without making any LLM calls** by default — it uses the routing decisions and the registered cost table to compute spend. Set `RUN_LIVE_QUALITY_CHECK = True` (and set `GOOGLE_API_KEY`) to additionally sample a quality check.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manthan9891994/agents-multi-model-support/blob/main/examples/benchmark_cost_savings.ipynb)

In [ ]:
!pip install -q 'dynamic-model-router[ml]' matplotlib pandas
import os
os.environ.setdefault('CLASSIFIER_TEST_MODE', '1')   # avoid touching cost-tracker globals

## 1. Load 1,000 prompts from a representative mix

In [ ]:
import random
random.seed(42)

# Representative prompt mix - proportions roughly match production agent traffic.
# Each bucket has 20-30 unique prompts to keep duplication low after sampling.
PROMPT_BUCKETS = {
    "casual_chat": [   # 25%
        "Hello, how are you today?", "Thanks!", "What is your name?",
        "Good morning", "Tell me a joke", "Have a nice day", "Who are you?",
        "Bye for now", "Cool, thanks", "Help me out", "Hi there", "Yo",
        "Are you AI?", "Got it", "Sounds good", "Awesome", "Sure thing",
        "Nice to meet you", "Take care", "Goodbye", "Whats up", "See you later",
        "Ok cool", "Maybe later", "Not now", "I disagree", "I agree",
        "Tell me more", "Continue", "Stop",
    ],
    "simple_qa": [
        "What is the capital of France?", "When did WWII end?",
        "Convert 100 USD to EUR", "What is 15% of 240?",
        "Define the word ephemeral", "Spell accommodate",
        "What year was Python released?", "How tall is Mount Everest?",
        "What is the speed of light?", "Largest country by area?",
        "How many continents are there?", "What does CPU stand for?",
        "Who painted the Mona Lisa?", "What is the boiling point of water?",
        "How many bones in human body?", "What is photosynthesis?",
        "Distance from earth to moon?", "Who wrote Hamlet?",
        "When was the internet invented?", "What is HTTP?",
        "How many planets in solar system?", "What is DNA?",
        "Largest ocean on Earth?", "How does gravity work?",
        "What is gluten?", "Who invented the telephone?",
        "What is GDP?", "How do vaccines work?",
        "What is renewable energy?", "How long is a marathon?",
    ],
    "simple_code": [
        "Write a Python function to reverse a string",
        "How do I read a CSV file in pandas?",
        "Show me a regex for an email address",
        "Fix this JavaScript: const x = 5; x++;",
        "How do I sort a list in Python descending?",
        "Convert this snake_case to camelCase: user_id, first_name",
        "Write a SQL query to select unique emails",
        "How do I create a dict in Python?",
        "What is f-string syntax in Python?",
        "How do I handle exceptions in Python?",
        "Write a regex for a phone number",
        "How to read a JSON file in JavaScript?",
        "How to merge two arrays in JavaScript?",
        "What is async/await?",
        "How do I deploy a Flask app?",
        "How to install a Python package?",
        "What is virtualenv?",
        "How do I git stash?",
        "Write a bash command to find files",
        "How to use sed to replace text?",
    ],
    "reasoning": [
        "Compare microservices vs monolith for a 10-engineer team building a B2B SaaS",
        "Why does the median income lag the mean income in most economies?",
        "Trade-offs of optimistic vs pessimistic locking in a multi-tenant DB",
        "Should we adopt Rust for our high-throughput payment service? Pros/cons.",
        "Compare PostgreSQL vs MongoDB for an e-commerce platform with 1M users",
        "Should we self-host or use managed Kubernetes for a 50-pod app?",
        "Pros and cons of GraphQL vs REST for a public API",
        "When does sharding make sense for a relational database?",
        "Should we write integration tests for every endpoint?",
        "Trade-offs of monorepo vs polyrepo for a 100-engineer org",
        "Why might a startup choose Go over Python in 2026?",
        "Compare server-side rendering vs client-side for a SaaS dashboard",
        "When does Cassandra outperform Postgres?",
        "Pros and cons of feature flags vs branching for releases",
        "Trade-offs of event sourcing vs CRUD for an audit-heavy domain",
        "Compare AWS Lambda vs Cloud Run for an API with bursty traffic",
        "Pros and cons of vector DB vs full-text search for RAG",
        "Should we migrate from REST to gRPC for internal services?",
        "Trade-offs of in-memory cache vs Redis for session storage",
        "When does HTAP outperform separate OLTP+OLAP databases?",
    ],
    "complex_doc": [
        "Write a 2,000-word technical RFC proposing a new event-sourced inventory system, including data model, CQRS commands, projection examples, migration plan from current state, and risk mitigation.",
        "Draft a comprehensive employment contract for a senior engineer with non-compete (NY-enforceable), IP assignment, severance, and arbitration clauses.",
        "Write a 1,500-word incident postmortem for an outage caused by a bad deployment, including timeline, root cause, contributing factors, action items, and prevention.",
        "Draft a vendor security questionnaire covering SOC 2, encryption, access control, data residency, and breach notification.",
        "Write a detailed migration plan from MySQL 5.7 to PostgreSQL 16 for a 2 TB OLTP database with 99.99% SLA.",
        "Draft a 3-page architectural decision record (ADR) for choosing Kafka over RabbitMQ in a fintech messaging pipeline.",
        "Write a comprehensive code style guide for a Python codebase with 500K LoC and 80 contributors.",
        "Draft a runbook for diagnosing high p99 latency in a multi-region Kubernetes deployment.",
        "Write a 2,000-word RFC for adopting OpenTelemetry across 40 microservices.",
        "Draft a HIPAA compliance plan for a clinical decision-support tool processing PHI in three regions.",
    ],
    "research": [
        "Design a complete distributed-systems architecture for a cardiology decision-support service handling 10K concurrent clinicians, with HIPAA compliance, multi-region failover, model A/B testing, audit trails, and a 5-year cost model.",
        "Build a comprehensive risk model for a transition from on-prem Oracle to Aurora PostgreSQL across 12 production databases serving 50K TPS.",
        "Design a multi-tenant LLM observability stack supporting 200 customers, 500K daily decisions, with per-tenant retention, cost attribution, and SOC 2 audit trails.",
        "Architect a real-time fraud detection platform processing 1M transactions/sec with sub-100ms decision latency, 99.99% uptime, and Basel III compliance.",
        "Design a complete data mesh implementation for a Fortune 500 retailer covering 200 source systems, federated governance, and contract-based interfaces.",
    ],
}

DIST = {"casual_chat": 0.25, "simple_qa": 0.30, "simple_code": 0.15,
        "reasoning":   0.15, "complex_doc": 0.10, "research": 0.05}

N_PROMPTS = 1000
prompts = []
for bucket, frac in DIST.items():
    n = int(N_PROMPTS * frac)
    prompts.extend(random.choices(PROMPT_BUCKETS[bucket], k=n))

random.shuffle(prompts)
unique_total = sum(len(v) for v in PROMPT_BUCKETS.values())
print(f"Loaded {len(prompts)} prompts ({unique_total} unique across buckets)")
print(f"Distribution: {DIST}")


## 2. Estimate baseline cost (always `gpt-4o`)

In [ ]:
from classifier.infra.tokenizers import count_tokens
from classifier.infra.cost_tracker import get_model_cost

BASELINE_MODEL = 'gpt-4o'
AVG_OUTPUT_TOKENS = 250          # typical agent response length

def cost_for(model: str, input_tokens: int, output_tokens: int = AVG_OUTPUT_TOKENS) -> float:
    rates = get_model_cost(model)
    return (input_tokens / 1e6) * rates['input'] + (output_tokens / 1e6) * rates['output']

baseline_cost = sum(cost_for(BASELINE_MODEL, count_tokens(p, model=BASELINE_MODEL)) for p in prompts)
print(f'Baseline (always {BASELINE_MODEL}): ${baseline_cost:.4f} for {len(prompts)} prompts')

## 3. Route each prompt and accumulate cost

In [ ]:
from classifier import Router
from collections import Counter

router = Router(layer2_enabled=False, layer3_enabled=True, cache_enabled=True)

routed_cost   = 0.0
tier_counts   = Counter()
model_counts  = Counter()
decisions     = []

for p in prompts:
    d = router.classify(p, provider='openai')   # provider=openai keeps cost units consistent
    tier_counts[d.tier.value]  += 1
    model_counts[d.model_name] += 1
    routed_cost += cost_for(d.model_name, count_tokens(p, model=d.model_name))
    decisions.append(d)

print(f'Routed cost: ${routed_cost:.4f}')
print(f'Savings:      ${baseline_cost - routed_cost:.4f} ({100*(1 - routed_cost/baseline_cost):.1f}%)')
print()
print('Tier distribution:')
for tier, count in sorted(tier_counts.items()):
    print(f'  {tier:6}: {count:4} ({100*count/len(prompts):4.1f}%)')
print()
print('Models used:')
for model, count in model_counts.most_common():
    print(f'  {model:30}: {count:4}')

## 4. The money chart — bar comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of total cost
ax = axes[0]
bars = ax.bar(['Always gpt-4o', 'Routed'], [baseline_cost, routed_cost],
              color=['#d62728', '#2ca02c'], width=0.55)
for bar, val in zip(bars, [baseline_cost, routed_cost]):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'${val:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Total cost (USD) for 1,000 prompts', fontsize=11)
savings_pct = 100 * (1 - routed_cost / baseline_cost)
ax.set_title(f'Cost comparison — {savings_pct:.1f}% savings', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Right: tier distribution pie
ax = axes[1]
tier_order = ['low', 'medium', 'high']
labels  = [f'{t.upper()} ({tier_counts[t]})' for t in tier_order]
sizes   = [tier_counts[t] for t in tier_order]
colors  = ['#2ca02c', '#ff7f0e', '#d62728']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
ax.set_title('Tier distribution across 1,000 prompts', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('cost_savings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart to cost_savings.png — copy to docs/img/cost_savings.png to embed in README')

## 5. Summary

This benchmark measures **routing decisions and cost arithmetic only** - it does not call any LLM by default. To validate quality parity (i.e. that the cheaper routed model produces equally good answers), pair this with your own eval suite (a hold-out set of (prompt, gold-answer) pairs scored by a frontier judge model). Sample 50-100 prompts from your real traffic, run both `gpt-4o` and the routed model, score with `gemini-2.5-pro` as judge.


| Metric | Typical value |
|---|---|
| Total prompts | 1,000 |
| Unique prompts | ~115 |
| Savings | **65-80%** depending on traffic mix |
| LOW tier hit rate | 55-65% |
| MEDIUM tier hit rate | 20-30% |
| HIGH tier hit rate | 8-15% |

**Caveats:**
- `AVG_OUTPUT_TOKENS = 250` is a heuristic. Real production traffic varies wildly per bucket - instrument your own outcome log to measure precisely.
- Cost figures use the bundled `default.yaml` registry (snapshot, ~Apr 2026). Override with your own registry to get current numbers.
- Routing accuracy depends on whether you've trained Layer 3 on your domain. Default L3 is zeroshot; expect higher savings after `dmr train --data your-data.jsonl`.

**To embed in your README:** copy `cost_savings.png` to `docs/img/cost_savings.png` and reference it.
